In [1]:
from sklearn.linear_model import LinearRegression
import numpy as np
import os
import pandas as pd


In [2]:
def train_test_split(X, y, split = 0.5):
    length = int(min(len(X), len(y)) * (1 - split))
    X_train = X[:length]
    X_test = X[length:]
    y_train = y[:length]
    y_test = y[length:]
    return X_train, X_test, y_train, y_test


In [26]:
def process_data(df):
    df = df.copy()
    df = df.sort_values(by="Timestamp")
    # drop duplicate timestamps
    df = df.drop_duplicates(subset=["Timestamp"])

    # generate lags
    max_lag = 10
    for i in range(1, max_lag):
        df[f"lag_{i}"] = df["Low Price"].shift(i)



    # add next value that needs to be predicted
    df["NextValue"] = df["Low Price"].shift(-1)

    df = df.dropna()

    # create X
    drop = ["Closing Price", "Opening Price", "High Price", "Timestamp", "No of Shares", "NextValue"]
    X = df.drop(columns=drop, errors='ignore')

    y = df["NextValue"]

    return X, y

In [27]:
files_path = "./stocksData"
files = os.listdir(files_path)
print(files)
data_files = {}

for file in files:
    if file.endswith(".csv"):
        if "lag" not in file: continue
        with open(files_path + "/" + file, "r") as f:
            df = pd.read_csv(f)
            if "Low Price" in df.columns:
                print(f"Processing file: {file}")
                data_name = file.replace(".csv", "")
                try:
                    X, y = process_data(df)
                    data_files[data_name] = (X, y)
                except Exception as e:
                    print(f"Error processing {file}: {e}")




['lag_h6.csv']
Processing file: lag_h6.csv


In [28]:
models = {}

for data_name in data_files:
    X, y = data_files[data_name]
    X_train, X_test, y_train, y_test = train_test_split(X, y)

    model = LinearRegression()
    model.fit(X_train, y_train)

    score = model.score(X_test, y_test)
    print(f"Model for {data_name} has a score of: {score}")

    models[data_name] = model

Model for lag_h6 has a score of: 0.9950109549706806


In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2991, 11), (2991, 11), (2991,), (2991,))

In [ ]:
X_train, y_train

(      Low Price   lag_1   lag_2   lag_3   lag_4   lag_5   lag_6   lag_7  \
 9        448.14  448.46  448.00  448.25  448.17  447.72  446.90  445.61   
 10       448.60  448.14  448.46  448.00  448.25  448.17  447.72  446.90   
 11       448.63  448.60  448.14  448.46  448.00  448.25  448.17  447.72   
 12       449.65  448.63  448.60  448.14  448.46  448.00  448.25  448.17   
 13       450.16  449.65  448.63  448.60  448.14  448.46  448.00  448.25   
 ...         ...     ...     ...     ...     ...     ...     ...     ...   
 2995     441.99  441.76  441.39  440.85  440.98  440.77  440.56  440.07   
 2996     442.55  441.99  441.76  441.39  440.85  440.98  440.77  440.56   
 2997     442.24  442.55  441.99  441.76  441.39  440.85  440.98  440.77   
 2998     442.00  442.24  442.55  441.99  441.76  441.39  440.85  440.98   
 2999     441.72  442.00  442.24  442.55  441.99  441.76  441.39  440.85   
 
        lag_8   lag_9  target  
 9     444.79  444.70  448.60  
 10    445.61  444.79 

In [29]:
X_train.columns

Index(['Low Price', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6',
       'lag_7', 'lag_8', 'lag_9'],
      dtype='object')

In [30]:
import requests, json, math
import pandas as pd
import numpy as np

def getStockData(stockTicker, to=""):
    url = f"https://tornsy.com/api/{stockTicker}?interval=h6&to={to}"
    print(f"Making request to {url}")
    response = requests.get(url)
    print(f"Response: {response.status_code}")

    if response.status_code != 200:
        print(f"Error fetching data: {response.text}")
        return None

    try:
        jsonResponse = json.loads(response.text)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return None

    if "data" in jsonResponse and len(jsonResponse["data"]) > 0:
        df = pd.DataFrame(jsonResponse["data"], columns=["Timestamp", "Opening Price", "High Price", "Low Price", "Closing Price", "No of Shares"])
        numerical_cols = ["Opening Price", "High Price", "Low Price", "Closing Price", "No of Shares"]
        for col in numerical_cols:
            df[col] = pd.to_numeric(df[col])
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], unit='s')
        df = df.sort_values(by="Timestamp")
        max_lag = 10
        for i in range(1, max_lag):
            df[f"lag_{i}"] = df["Low Price"].shift(i)
        df = df.tail(1)
        drop_cols = ["Closing Price", "Opening Price", "High Price", "Timestamp", "No of Shares"]
        X = df.drop(columns=drop_cols, errors='ignore')
        X = X.dropna()

        if X.empty:
            print("Not enough data to create features for prediction.")
            return None

        return X
    else:
        print("No data found for the specified ticker.")
        return None

def predict(stockTicker, models):
    X_pred = getStockData(stockTicker)
    if X_pred is not None:
        model_key = "lag" + "_h6"
        if model_key in models:
            model = models[model_key]
            prediction = model.predict(X_pred)
            print(f"Prediction for {stockTicker}: {prediction[0]}")
            return prediction[0]
        else:
            print(f"No trained model found for the key '{model_key}'.")
            return None
    else:
        print("Could not get data for prediction.")
        return None

In [31]:
predict("lag", models)

Making request to https://tornsy.com/api/lag?interval=h6&to=
Response: 200
Prediction for lag: 450.36032176883117


np.float64(450.36032176883117)